In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

In [ ]:
load_dotenv()

In [ ]:
model = init_chat_model(
    model="gpt-4o",
    model_provider="openai",
    temperature=0,
    timeout=30,
    max_retries=3,
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [("system", "You are a helpful assistant"), ("human", "{input}")]
)

chain = prompt | model

response = chain.invoke({"input":"سلام خوبی؟"})
response

## Streaming

In [ ]:
for chunk in model.stream("یه پاراگراف در مورد شرکت بنز بگو"):
    print(chunk.content, end="", flush=True)

In [ ]:
# batch — ارسال چند request موازی
responses = model.batch(
    [
        "سلام",
        "من آرش هستم",
        "امروز چندمه",
    ]
)
for i, r in enumerate(responses):
    print(f"Response {i+1}: {r.content[:80]}...")

## Pydantic

In [ ]:
from pydantic import BaseModel, Field


class Product(BaseModel):
    """اطلاعات استخراج‌شده از یک محصول"""

    name: str = Field(description="عنوان محصول")
    price: float = Field(description="قیمت محصول")
    options: list[str] = Field(description="نکات مثبت و منفی محصول")


product = "این گوشی سامسونگ مدل گلکسی اس 25 هست. خیلی گوشی خوش دست و سبکی هست. بسیار گرونه و شارژر هم نداره. قیمتش حدود 300 میلیون تومنه"

structured_model = model.with_structured_output(Product)
response = structured_model.invoke(f"اطلاعات این محصول رو استخراج کن {product}")
response

## Memory

In [ ]:
from langchain.agents import create_agent
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=[],
    checkpointer=InMemorySaver(),
    system_prompt="تو یک دستیار هوش مصنوعی هستی که جواب سوالات رو با صمیمیت و لحن کوچه بازاری جواب میدی",
)
config: RunnableConfig = {"configurable": {"thread_id": "chat-1"}}

agent.invoke(
    {"messages": [HumanMessage(content="اسم من آرش")]},
    config=config,
)

response = agent.invoke(
    {"messages": [HumanMessage(content="اسم من چیه؟")]}, config=config
)
print(response["messages"][-1].content)

### Memory Content

#### مشاهده تاریخچه مکالمه

In [ ]:
state = agent.get_state(config)
for message in state.values["messages"]:
    print(message)

### Middleware

In [ ]:
# نگه داشتن 4 پیام آخر
from typing import Any

from langchain.agents import AgentState
from langchain.agents.middleware import before_model
from langchain.messages import RemoveMessage
from langchain_core.runnables import RunnableConfig
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.runtime import Runtime


@before_model
def remove_messages(state: AgentState, runtime: Runtime):
    """فقط 4 پیام آخر رو نگه می‌داره"""

    messages = state["messages"]
    if len(messages) < 4:
        return None
    recent = messages[-4:]

    return {"messages": [RemoveMessage(id=REMOVE_ALL_MESSAGES), *recent]}


agent = create_agent(
    model=model,
    middleware=[remove_messages],
    checkpointer=InMemorySaver(),
    system_prompt="You are a helpful assistant.",
)
config_w: RunnableConfig = {"configurable": {"thread_id": "window-1"}}


agent.invoke(
    {"messages": [{"role": "user", "content": "Hi, my name is Arash"}]},
    config=config_w,
)
agent.invoke(
    {"messages": [{"role": "user", "content": "What is 1+1?"}]}, config=config_w
)
agent.invoke(
    {"messages": [{"role": "user", "content": "What is 2+2?"}]}, config=config_w
)

# این سوال آخر —  اسم رو به یاد ندارد  چون trim شده
response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]}, config=config_w
)
print(response["messages"][-1].content)

In [ ]:
# نگه داشتن 4 پیام آخر

from langchain.agents import AgentState
from langchain.agents.middleware import after_model
from langchain_core.runnables import RunnableConfig
from langgraph.runtime import Runtime


@after_model
def remove_old_message(state: AgentState, runtime: Runtime):
    """بعد از هر پاسخ، پیام‌های قدیمی رو حذف می‌کنه"""
    messages = state["messages"]

    if len(messages) >= 4:
        # 2 پیام اول رو حذف کن
        return {"messages": [RemoveMessage(id=m.id) for m in messages[:2]]}
    return None


agent = create_agent(
    model=model,
    tools=[],
    middleware=[remove_old_message],
    checkpointer=InMemorySaver(),
)

config_d: RunnableConfig = {"configurable": {"thread_id": "delete-1"}}
agent.invoke(
    {"messages": [{"role": "user", "content": "Hi, my name is Ali"}]}, config=config_d
)
response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]}, config=config_d
)
print(response["messages"][-1].content)

### SummarizationMiddleware

جایگزین `ConversationSummaryBufferMemory` قدیمی.
وقتی تعداد توکن‌ها از حد گذشت، مکالمات قدیمی رو خلاصه می‌کنه.

In [ ]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.runnables import RunnableConfig

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 200),  # وقتی از 200 توکن گذشت، خلاصه کن
            keep=("messages", 4),  # 4 پیام آخر رو نگه دار
        )
    ],
    checkpointer=InMemorySaver(),
    system_prompt="You are a helpful assistant.",
)

config_s: RunnableConfig = {"configurable": {"thread_id": "summary-1"}}

agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """
من دارم روی یک فروشگاه اینترنتی کار می‌کنم.
اسم پروژه Shopino هست.
بک‌اند با Python و FastAPI نوشته شده.
دیتابیس PostgreSQL هست.
برای کش از Redis استفاده می‌کنیم.
فرانت‌اند هم React هست.
قرار شده نسخه اول پروژه تا آخر ماه آماده بشه.
""",
            }
        ]
    },
    config=config_s,
)

agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """
تیم ما 4 نفره است.
من روی بک‌اند کار می‌کنم.
علی مسئول فرانت‌اند است.
رضا روی DevOps کار می‌کند.
سارا هم مسئول طراحی UI/UX است.
سرور اصلی روی AWS قرار دارد.
""",
            }
        ]
    },
    config=config_s,
)

agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """
امروز تصمیم گرفتیم سیستم احراز هویت با JWT پیاده‌سازی شود.
توکن access بیست دقیقه اعتبار داشته باشد.
refresh token هفت روز اعتبار داشته باشد.
همچنین کاربران بتوانند با گوگل هم وارد شوند.
""",
            }
        ]
    },
    config=config_s,
)

agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """
برای بخش محصولات هم تصمیم گرفتیم هر محصول
نام، قیمت، توضیحات، دسته‌بندی و موجودی داشته باشد.
تصاویر محصولات در S3 ذخیره شوند.
جستجوی محصولات هم بعداً با Elasticsearch پیاده‌سازی شود.
""",
            }
        ]
    },
    config=config_s,
)

response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "تا الان چه تصمیم‌هایی برای پروژه گرفته بودیم؟"}
        ]
    },
    config=config_s,
)

print(response["messages"][-1].content)

In [ ]:
# مشاهده وضعیت حافظه بعد از خلاصه‌سازی
state = agent.get_state(config_s)

for msg in state.values["messages"]:
    print("=" * 50)
    print(type(msg).__name__)
    print(msg.content)

### Database Memory

In [ ]:
import sqlite3

from langchain.agents import create_agent
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.sqlite import SqliteSaver

# اتصال به SQLite
# اگر فایل وجود نداشته باشه، خودش ساخته میشه
conn = sqlite3.connect(
    "checkpoints.sqlite",
    check_same_thread=False,
)

# ساخت Checkpointer
checkpointer = SqliteSaver(conn)


# ساخت Agent
agent = create_agent(
    model=model,
    tools=[],
    checkpointer=checkpointer,
    system_prompt="تو یه دستیار حرفه ای چت هستی",
)


# شناسه مکالمه
config: RunnableConfig = {"configurable": {"thread_id": "user-568"}}


# پیام اول
response = agent.invoke(
    {"messages": [{"role": "user", "content": "اسم من آرشه"}]},
    config=config,
)

print(response["messages"][-1].content)


# پیام دوم — باید اسم رو از حافظه بدونه
response = agent.invoke(
    {"messages": [{"role": "user", "content": "اسم من چیه؟"}]},
    config=config,
)

print(response["messages"][-1].content)